<a href="https://colab.research.google.com/github/Amrutha-K-Mohanan/EmailSpamDetection/blob/1-spam-detection-model/Email_Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas scikit-learn nltk

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
data = pd.read_csv("spam.csv", encoding="latin-1")
data = data[['v1', 'v2']]
data.columns = ['label', 'message']

print(data.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [16]:
nltk.download('stopwords')
ps = PorterStemmer()

corpus = []

for msg in data['message']:
    review = re.sub('[^a-zA-Z0-9]', ' ', msg)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if word not in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# tfidf = TfidfVectorizer(max_features=3000)
# X = tfidf.fit_transform(corpus).toarray()

# y = data['label'].map({'ham': 0, 'spam': 1})

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000)
X = cv.fit_transform(corpus).toarray()
y = data['label'].map({'ham': 0, 'spam': 1})

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
model = MultinomialNB(alpha=0.5)
model.fit(X_train, y_train)


MultinomialNB(alpha=0.5)

In [20]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9820627802690582

Confusion Matrix:
 [[954  11]
 [  9 141]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.93      0.94      0.93       150

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [29]:
def predict_spam(email):
    email = re.sub('[^a-zA-Z0-9]', ' ', email)
    email = email.lower().split()
    email = [ps.stem(word) for word in email if word not in stopwords.words('english')]
    email = ' '.join(email)

    email_vector = cv.transform([email]).toarray()
    prediction = model.predict(email_vector)

    return "Spam" if prediction[0] == 1 else "Not Spam"

# Example
print(predict_spam("Hey, are we still meeting today?"))

Not Spam


In [30]:
import pickle

In [31]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

In [32]:
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(cv, f)

In [36]:
from google.colab import files

In [34]:
files.download("model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
files.download("vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>